<a href="https://colab.research.google.com/github/jeffheaton/app_generative_ai/blob/main/t81_559_class_07_2_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-559: Applications of Generative Artificial Intelligence
**Module 7: LangChain: Agents**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://github.com/jeffheaton/app_generative_ai).

# Module 7 Material

* Part 7.1: Introduction to LangChain Agents [[Video]](https://www.youtube.com/watch?v=J5Vr___lSSs) [[Notebook]](t81_559_class_07_1_agents.ipynb)
* **Part 7.2: Understanding LangChain Agent Tools** [[Video]](https://www.youtube.com/watch?v=qMquBmteYw4) [[Notebook]](t81_559_class_07_2_tools.ipynb)
* Part 7.3: LangChain Retrival and Search Tools [[Video]](https://www.youtube.com/watch?v=NB5qGPLoBBE) [[Notebook]](t81_559_class_07_3_search_tools.ipynb)
* Part 7.4: Constructing LangChain Agents [[Video]](https://www.youtube.com/watch?v=OJe5oHvrdHk) [[Notebook]](t81_559_class_07_4_more_agent.ipynb)
* Part 7.5: Custom Agents [[Video]](https://www.youtube.com/watch?v=IsJemVYSEdc) [[Notebook]](t81_559_class_07_5_custom_agent.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.

In [ ]:
import os

try:
    from google.colab import drive, userdata
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# OpenAI Secrets
if COLAB:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Install needed libraries in CoLab
if COLAB:
    !pip install langchain langchain_openai

# 7.2: LangChain Agent Tools


LangChain agents are versatile entities designed to perform specific tasks autonomously. Central to their functionality are [tools](https://docs.langchain.com/oss/python/langchain/tools), which are specialized components that agents can utilize to accomplish their objectives. These tools can range from data retrieval and processing utilities to interactive interfaces for user engagement. By leveraging these tools, LangChain agents can efficiently execute complex workflows, automate routine tasks, and provide intelligent solutions tailored to user needs. Whether it's querying databases, parsing documents, or interacting with APIs, the strategic use of tools enables LangChain agents to enhance productivity and deliver precise outcomes.


Large Language Models (LLMs) inherently lack access to real-time information such as the current date and time, stock market data, and breaking news. This limitation stems from their design, which relies on pre-existing datasets that do not include ongoing updates. Additionally, LLMs are not equipped to perform mathematical calculations directly, which can restrict their utility in scenarios requiring precise numerical operations. To overcome these constraints, LLMs can employ specialized tools. For instance, search engine tools enable LLMs to retrieve the latest information from the web, ensuring up-to-date responses. Similarly, tools designed for mathematical computation can assist LLMs in accurately processing and solving mathematical problems, thereby enhancing their overall capability and accuracy.

* [Available Langchain Tools](https://python.langchain.com/v0.2/docs/integrations/tools/)
* [Available Langchain Toolkits](https://python.langchain.com/v0.2/docs/integrations/toolkits/)

## Tools for Math


We will begin our exploration of LangChain tools by using a tool specifically designed for math. Large Language Models (LLMs), like ChatGPT, are generally poor at performing mathematical calculations without the assistance of specialized tools. This is because LLMs are trained primarily on textual data and lack the precision required for accurate arithmetic operations. Consequently, they often produce inaccurate results when asked to perform math independently. To see how a LLM actually performs mathematics, I asked ChatGPT and it gave a decent high-level summary.

> Large Language Models (LLMs), like me, do not inherently perform arithmetic calculations the same way a calculator or dedicated algorithm would. Instead, we generate responses based on patterns in the data we've been trained on. Here's a simplified explanation of how we handle such tasks:
>
> Pattern Recognition: During training, LLMs are exposed to vast amounts of text data, which includes examples of arithmetic and mathematical reasoning. We learn patterns and structures in these examples, enabling us to approximate calculations.
>
> Token Prediction: When asked to perform a calculation, an LLM doesn't actually "calculate" in the traditional sense. Instead, it predicts the most likely sequence of tokens (numbers, in this case) that should follow based on the input. This prediction is influenced by the training data but does not involve real arithmetic operations.
>
> Approximation and Heuristics: For smaller or simpler calculations, the model might generate the correct answer because it has seen enough examples during training. For larger or more complex calculations, the model might generate an approximate answer or even make a guess based on learned patterns.
>
> For example, if you ask an LLM to multiply 872947493 by 7492374932, it will try to generate a plausible sequence of digits based on what it has seen in the training data, but this sequence is unlikely to be correct without an actual computational algorithm.
>
>Here’s a brief comparison of how a traditional method (e.g., a calculator or algorithm) and an LLM approach such a problem:
>
> * Traditional Method: Uses precise algorithms to perform each step of the multiplication (e.g., long multiplication or fast algorithms like the Karatsuba algorithm).
> * LLM Method: Predicts the next sequence of digits based on patterns and probabilities from the training data.
So, while an LLM might "attempt" to give an answer, it lacks the precision and algorithmic foundation to guarantee accuracy for complex arithmetic without dedicated computational tools.

To see this in action, let's ask an LLM to perform a mathematical operation. We will choose numbers that were unlikely in the LLM's training data.



In [2]:
from langchain_openai import ChatOpenAI

MODEL = "gpt-5.6-luna"

llm = ChatOpenAI(
        model=MODEL,
        use_responses_api=True  # tool calling on gpt-5.6 models requires the Responses API
    )

print(llm.invoke("What is 8273 times 1821?"))

content=[{'id': 'rs_077398a80bb2633a006a6dde3cf9d48194b0117a3cc83be529', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqbd49_qdTIgHKaSaaClXHAeH5LNnYTrJQn5NfQzVt9FifggGKsLXGT8k79hxyDnXLGwpvxOEZxYDO4Cundjg5bilfy4vksImDG4Xfeoh9u7WrL4ANALMBN1vFUqlyuLy8kLPrF-6AHCCnng-oybt5U-O-HY_-BvoVDjNbf2BypLBI6Q-n-VIpDdApW9qop79jU2HtQ1k0cxuORrsNa7nkn5tPnAZ9BG27mKbjRTEErmquch1HSHC7B3m-dlLTcr_aTZNYIpx2_CVqutkph7VarWQLd_K7VlbgwMjzXgvIDQ-OKaukeoMJagZA0h9pzea9Lu3jF9-mc_zwervctfK98PBS3x8_txU_RBBYdmnBkDC0JToFTWtUq0ntlZoV6BI888tZjMBguqykMvlktBixH8vK4yJ4oSYITgUEtX6-8vxCck1ztRSvJhYHv_4YwDkn7EeiYh6E2_nHWiRv-kqaK9-1kQLicWobRgougWzfHabGucrdVeLHo68e6omwBCj_2ZXZ9tAQ9sVw4MdeRb6bXcfY-fCZTMDAv4Rn2orwngY_zYD4jq9rR86TLxvzo3-6Qi3xZNc49sGtUMubEg8YN8B2knG335YbG6dfppFFgw5V5fZbXTv71KXqlPs5paXyt3uFjPIoMl-CglR8l86JQt-QhanuDO9OfL3AT3VG8OHKBcy2Y_VA2Z2puidNmvDv-iVsSQ-Oa2BwtowRYr964MtvJmNzoFMJGjtTFx7QRTiqaeI9STLCmRe46YzMMAxYQKRe3q3udMDx_XnlMfXS5cjNppfDrYJAoropL02u28ZnVmLwrbOiVp6c3ZtvHHZdIyyBCjcak92Uw2

The resulting number appears reasonable, but that's the point. LLMs are trained to produce believable results, not necessarily correct ones. To verify the LLM, we'll have Python perform this calculation.

In [3]:
print(8273 * 1821)

15065133


In [4]:
print( abs(15065133 - 15055433 ) )


9700


We can see that the LLM was several thousand off. Unfortunately, LangChain does not include a calculator tool, at least as of 2026. We will look at two approaches. First, we will build a tiny Python execution utility: a few lines around Python's exec function that capture whatever the code prints. Utilities like this once shipped in LangChain's experimental package; writing it ourselves shows how little magic is involved, and how much power a tool like this hands to the model.

In [ ]:
import io
import contextlib

def run_python(code: str) -> str:
    """Execute Python code and return anything it prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        exec(code)
    return buffer.getvalue()

run_python("print(1+1)")

The following code builds an agent that can use a Python shell as a tool. A tool for executing Python commands (repl_tool) is created using the StructuredTool class, with a Pydantic schema (PythonReplInput) describing the single string argument and a description indicating its purpose as a Python shell that requires valid Python commands. The function run_python_code is assigned to execute the commands.

The agent itself is created with the create_agent function, which takes the language model (llm) and the list of tools (here, just repl_tool) and returns a ready-to-run agent; no separate prompt template or executor object is needed. We stream the agent's execution so you can see each step: the model deciding to call the Python tool, the tool's output, and the final answer.

In [6]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import StructuredTool
from pydantic import BaseModel

llm = ChatOpenAI(
    model=MODEL,
    use_responses_api=True  # tool calling on gpt-5.6 models requires the Responses API
)

# Define a Pydantic schema for the input arguments
class PythonReplInput(BaseModel):
    input: str

# Define the tool using StructuredTool with an args_schema
def run_python_code(input: str, **kwargs):
    try:
        # Safely evaluate the input string as a Python expression
        result = eval(input)
        return str(result)
    except Exception as e:
        return str(e)

repl_tool = StructuredTool(
    name="python_repl",
    description="A Python shell. Use this to execute python commands. Input should be a valid python command. If you want to see the output of a value, you should print it out with `print(...)`.",
    func=run_python_code,  # Use the custom Python REPL function
    args_schema=PythonReplInput  # Define the expected input schema
)

tools = [repl_tool]
agent = create_agent(llm, tools)

for step in agent.stream(
    {"messages": [{"role": "user", "content": "What is 8273 * 1821?"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()


================================ Human Message =================================

What is 8273 * 1821?
================================== Ai Message ==================================

[{'id': 'rs_050e30098ea62f5a006a6dde405d348196860804ffb4513824', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqbd5AOFIQ5Zds7Xwa8w2LfkhQXnBRRm5qKBogLn8HDKGUVdZCZmzVlnnjNGQ5BYF3GDwgb_GVRFLPMRIJG1mtN0QDucljluCc8b4UfzD1DGgmxI59d3AX1pOcBIKRjCZpNKPPAIqRzabJiYTUbMHMfe8G_IS_o9vV1vYVcKiT0scSOvNFXZxyRbABATg2ey0YeW7fEVvTVN4X8EY1urXYEZ_-A3JLfNPOcnVtifSNJPd_sOSCwMfYHtVmKij5gt5wRRkDxR4MDgs6WlDA0H6IwiY0PmK83wRgWYq1UqnA-CwlGMQugeHUXVEKWqnNI6oXUHXV7hH6qU9I07iybNK_W2JsffI6PSd1Cx4Pbv7PQqETtToJuGkkoUgbQ9ftStZ1CIeZvG7enwPvO6ydsKoBq4cuoIjNXa_Ce9fU9RJlXRSkiKgeg6gFrCCogxt0NsYjDExsxgcBklWzaugHJIqSVBkm7EkS_dEDJEjhq1iItKx_jS_YMhqtUlyG-OU-jrE8u0OU2xDFHGPAYVerawAfC3UVp90nMl57yzYR3ULFotEjqBLOMMlDp5CDk_6xo6TReUDFH0H_W0tJaKVG8wnw6cJ7rekn_gCwdUDYg0Qt1dyLeFjfLhhGXs1Q70JnFl00w72FX_U2VJi8IqKeOyRRbmvbFjIoC

## Create a Cusom Math Tool

The Python REPL tool we just used can execute any Python command. Therefore, it can be a security concern if we only wish to perform math calculations. In this section, we will see how to create a custom tool that can only perform basic math calculations.

In [7]:
import numpy as np

class SafeCalculator:
    def calculate(self, expression):
        try:
            # Evaluate the mathematical expression using NumPy
            result = eval(expression, {"__builtins__": None}, {"np": np})
            return result
        except Exception as e:
            return str(e)

# Initialize the safe calculator tool
safe_calculator = SafeCalculator()

# Example usage
expression = "3 * (2 + 5) / 7"
result = safe_calculator.calculate(expression)
print(f"Result: {result}")


Result: 3.0


In [8]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import StructuredTool
from pydantic import BaseModel

llm = ChatOpenAI(
    model=MODEL,
    use_responses_api=True  # tool calling on gpt-5.6 models requires the Responses API
)

# Define a Pydantic schema for the input arguments
class MathExpressionInput(BaseModel):
    input: str

# The tool calls the SafeCalculator we defined above
def safe_math(input: str, **kwargs):
    return str(safe_calculator.calculate(input))

# Define the tool using StructuredTool with args_schema
safe_math_tool = StructuredTool(
    name="safe_calc",
    description="A math calculator used to evaluate mathematical expressions. Input should be a valid math expression, similar to Python.",
    func=safe_math,  # Use the safe calculator function
    args_schema=MathExpressionInput  # Define the expected input schema
)

tools = [safe_math_tool]
agent = create_agent(llm, tools)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "8273 * 1821"}]}
)
print(result["messages"][-1].content)


[{'type': 'text', 'text': '15,065,133', 'annotations': [], 'id': 'msg_0fe8b88c7418c641006a6dde43b48081908d9b6071fca5b369', 'phase': 'final_answer'}]
